# Round 1 · Rules ⚔️ tier A

Write detections an analyst could defend in one sentence each. Each rule gets you a list of `(account_id, event_id)` pairs; the union is `suspects_r1.csv`. Score it as often as you like: `score_now()` prints recall per (unnamed) campaign, precision at queue sizes, and **every legitimate cohort you flagged against its 2% budget**. Names come at the reveal.

**The five rules to try** (content doc §7.3.1): impossible travel · new context + high-risk change · stuffed then entered · MFA fatigue · reset from nowhere.

**One clause every rule needs:** "new device / new network" means new relative to history *the log actually contains*. The log starts on 1 October; on an account's first day in it, everything is new. Require some earlier events on the account before you believe novelty.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../tools"))
from atolab import *
import matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

df = dedupe(load_events())
df = first_seen(df)                                   # is_new_device_id, is_new_asn, is_new_country, is_new_ja4, is_new_ip
df["n_prior"] = df.groupby("account_id", observed=True).cumcount()   # how much history the log has for this account at this event
HIGH = ["password_change", "email_change_confirmed", "mfa_disabled", "recovery_email_change", "phone_change"]
alerts_off = (df.event_type == "notification_settings_changed") & (df.d_alerts_enabled == False)
high_risk_by_user = (df.event_type.isin(HIGH) | alerts_off) & (df.actor == "user")
MIN_HISTORY = 10

## 1.1 A helper: "trigger followed by target within N hours" on the same account

In [ ]:
def followed_within(df, trigger_mask, target_mask, hours):
    trig = df.loc[trigger_mask, ["account_id", "ts", "event_id"]]
    targ = df.loc[target_mask, ["account_id", "ts"]].rename(columns={"ts": "t_ts"})
    m = trig.merge(targ, on="account_id")
    m = m[(m.t_ts >= m.ts) & (m.t_ts <= m.ts + pd.Timedelta(hours=hours))]
    return m.drop_duplicates("event_id")[["account_id", "event_id"]]

## 1.2 Rule: new context + high-risk change

A login from a device **and** a network the account has never used, followed within an hour by a user-initiated high-risk change. Start here; it is the rule people write first, and the scorer will show you what it costs. Then try the same rule without `MIN_HISTORY`, and with the medium-risk events (`address_added`, `payment_method_added`, `order_placed`) added to the target set.

In [ ]:
trig = (df.event_type == "login_success") & df.is_new_device_id & df.is_new_asn & (df.n_prior >= MIN_HISTORY)
r_newctx = followed_within(df, trig, high_risk_by_user, hours=1).assign(rule="new_context_high_risk")
print(len(r_newctx), "hits on", r_newctx.account_id.nunique(), "accounts")

## 1.3 Rule: impossible travel

Consecutive logins from different countries, too far apart for the time in between (`with_geo` joins city centroids; `haversine_km` does the rest). Think about what *else* the rule needs so the three-day geo-IP glitch you found in Round 0 does not produce 150 findings.

In [ ]:
logins = with_geo(df[(df.event_type == "login_success") & (df.n_prior >= MIN_HISTORY)]).sort_values(["account_id", "ts"])
g = logins.groupby("account_id", observed=True)
logins["km"] = haversine_km(g.lat.shift(), g.lon.shift(), logins.lat, logins.lon)
hours = (logins.ts - g.ts.shift()).dt.total_seconds() / 3600
logins["kmh"] = logins.km / hours.clip(lower=1 / 60)
cand = logins[(logins.country.astype(str) != g.country.shift().astype(str)) & (logins.km > 500) & (logins.kmh > 900)]
# TODO: add the clause that separates travel from a geo-IP table change, then require a high-risk change within 2 h
r_travel = followed_within(df, df.event_id.isin(cand.event_id), high_risk_by_user, hours=2).assign(rule="impossible_travel")
print(len(cand), "impossible-travel candidates ->", r_travel.account_id.nunique(), "accounts with a high-risk change after")

## 1.4 Rule: stuffed, then entered

The loud campaign fails a lot before it succeeds. A bad-password failure from datacenter/hosting space, then a password login from datacenter space on the same account. No sensitive action required: this is the only rule that can see an **access-only** takeover (login, read the payment methods, leave).

In [ ]:
dc = df.asn_type == "datacenter"
fails = df[(df.event_type == "login_failure") & dc][["account_id", "ts"]].rename(columns={"ts": "f_ts"})
dlog = df[(df.event_type == "login_success") & dc & (df.auth_method == "password")][["account_id", "ts", "event_id"]]
m = dlog.merge(fails, on="account_id")
m = m[(m.f_ts < m.ts) & (m.f_ts >= m.ts - pd.Timedelta(hours=48))]
r_stuffed = dlog[dlog.event_id.isin(m.event_id)][["account_id", "event_id"]].assign(rule="stuffed_then_entered")
print(r_stuffed.account_id.nunique(), "accounts")

## 1.5 Your turn: MFA fatigue, reset from nowhere

- **MFA fatigue:** five or more `mfa_challenge_failure` with `mfa_method == "push"` inside fifteen minutes, then a success.
- **Reset from nowhere:** a user-initiated `password_reset_requested` from a device *and* country the account has never used, completed, and followed by a high-risk change within the hour. (Forgetful owners reset from their own devices.)

In [ ]:
# TODO
r_fatigue = pd.DataFrame(columns=["account_id", "event_id", "rule"])
r_reset = pd.DataFrame(columns=["account_id", "event_id", "rule"])

## 1.6 Submit and score

The first event per account is your *time-to-detect* claim; the scorer grades it. Rank by how many rules fired.

In [ ]:
hits = pd.concat([r_newctx, r_travel, r_stuffed, r_fatigue, r_reset], ignore_index=True).merge(df[["event_id", "ts"]], on="event_id").sort_values("ts")
first = hits.groupby("account_id", observed=True).agg(first_event_id=("event_id", "first"), n_rules=("rule", "nunique")).reset_index().sort_values("n_rules", ascending=False)
save_suspects(first[["account_id"]], "../suspects_r1.csv", scores=first.n_rules, first_event=first.first_event_id)
score_now("../suspects_r1.csv")

## 1.7 Before Round 2

- Which cohorts did your first draft hurt, and which clause fixed it?
- Which tier is untouched, and why would *no* per-account rule reach it?
- Try `--wide`: add the medium-risk events to the target set. What does it buy, and from whom?

Fell behind? `python tools/checkpoint.py --round 1 --catch-up` writes the reference `suspects_r1.csv` so Round 2 can start.